# Stochastic and random forcing

The package also handles a forcing with a random component, for the equation

$$
\mathrm{d}\mathbf{u} + \left(\mathbf{u}\cdot\nabla\mathbf{u} + \nabla p\right)\mathrm{d}t = \nu\Delta\mathbf{u}\,\mathrm{d}t + \mathbf{f}\,\mathrm{d}t + N_t\,\mathrm{d}t + \sigma\Phi\,\mathrm{d}W_t, \qquad \nabla\cdot\mathbf{u} = 0,
$$

where $\mathbf{f}$ is a steady forcing, as before, $W_t$ is an $n$-dimensional Wiener process, $\Phi : \mathbb{R}^n \to H$ maps it onto divergence-free velocity fields (the modes forced stochastically), and $N_t$ is an arbitrary stochastic process with values in $H$, such as an Ornstein-Uhlenbeck process, a compound Poisson process, or the output of an embedded low-dimensional chaotic system. Each term may be absent. In vorticity form, the curls of these terms are added to the equation.

* The forcing modes are given by `fourier_modes(grid, κmin, κmax)` (or by a list of wavenumbers, or by arbitrary fields with `ForcingModes`): divergence-free Fourier modes $\sqrt{2}\,\nabla^\perp(\cos(\boldsymbol{\kappa}\cdot\mathbf{x})/|\boldsymbol{\kappa}|)$ and $\sqrt{2}\,\nabla^\perp(\sin(\boldsymbol{\kappa}\cdot\mathbf{x})/|\boldsymbol{\kappa}|)$, normalized to unit mean square.
* The stochastic forcing $\sigma\Phi\,\mathrm{d}W_t$ is an `AdditiveNoise(modes; σ, amplitudes)`, with $\Phi e_j$ = `amplitudes[j]` times the $j$-th mode. Its mean injection rates of energy, $\varepsilon = \tfrac12\sigma^2\sum_j a_j^2\langle|\boldsymbol{\phi}_j|^2\rangle$, and of enstrophy, $\eta = \tfrac12\sigma^2\sum_j a_j^2\langle|\nabla\times\boldsymbol{\phi}_j|^2\rangle$, do not depend on the flow (Itô), and are given by `energy_injection_rate` and `enstrophy_injection_rate`.
* The random forcing $N_t = \sum_j \eta_j(t)\boldsymbol{\phi}_j$ is a `RandomForcing(modes, η)`, for any function `η(t)` returning the coefficients: sample paths from `ou_path`, `compound_poisson_path` and `lorenz_path`, or any other.

With a stochastic forcing, problems are built with `kind = :sde`, for the solvers of [StochasticDiffEq.jl](https://github.com/SciML/StochasticDiffEq.jl) (e.g. `EM()`, `SRA1()`, `SOSRA()`), or integrated with the fixed-step schemes of the package: `StochasticETDRK4()`, `StochasticExpEuler()` and `SplitStochasticRK(:rk4)` or `SplitStochasticRK(:ssprk3)`. A random forcing alone is a time-dependent forcing, and the problem can be solved with any of the deterministic methods of the previous pages.

## Packages

In [ ]:
using NSEPeriodic2D
using OrdinaryDiffEq: Tsit5
using Plots
using Printf
using Random: Xoshiro

plotfield(grid, f; kwargs...) = heatmap(grid.x, grid.y, permutedims(f);
    aspect_ratio = 1, xlims = (0, grid.Lx), ylims = (0, grid.Ly), xlabel = "x", ylabel = "y",
    color = :balance, clims = (-1, 1) .* maximum(abs, f), titlefont = 10, kwargs...)
nothing

## A flow driven by an additive noise

A $64 \times 64$ grid, viscosity $\nu = 10^{-2}$, and a white-in-time forcing on all the modes with $3 \leq |\boldsymbol{\kappa}| \leq 5$, with energy injection rate $\varepsilon = 0.1$ and no deterministic forcing:

In [ ]:
grid = Grid(Lx = 2π, Nx = 64)
modes = fourier_modes(grid, 3, 5)
ε = 0.1
noise = AdditiveNoise(modes; σ = sqrt(2ε / length(modes)))

In [ ]:
model = NSEModel(grid; ν = 1e-2, noise)

Starting from rest, we integrate with the stochastic ETDRK4 scheme, with a fixed step:

In [ ]:
prob = NSEProblem(model, zeros(grid.Nx, grid.Ny), (0.0, 400.0); kind = :sde)
res = simulate(prob, StochasticETDRK4(); dt = 0.01, save_every = 0.5, seed = 2026)

In [ ]:
plot(res.t, [res.energy res.enstrophy], xlabel = "t", label = ["E" "Z"],
    title = "Energy and enstrophy, driven by the noise", titlefont = 10)

In a statistically steady state, the mean energy and enstrophy budgets close exactly: $2\nu\langle Z\rangle = \varepsilon$ and $2\nu\langle P\rangle = \eta$, with $Z$ the enstrophy and $P$ the palinstrophy. This holds for any flow, chaotic or not, so it is a check of a numerical method that does not depend on the trajectory. After the transient, over this short window, the ratios below should be within a few percent of 1, the size of the sampling error (they change from one build of this page to another); see the comparison of methods at the end for longer runs:

In [ ]:
s = res.t .> 150
@printf("2ν⟨Z⟩/ε = %.3f,   2ν⟨P⟩/η = %.3f\n", 2 * model.ν * sum(res.enstrophy[s]) / count(s) / ε,
    2 * model.ν * sum(res.palinstrophy[s]) / count(s) / enstrophy_injection_rate(noise))

In [ ]:
plotfield(grid, vorticity(model, res.w_hat_final); title = "vorticity at t = 400")

## Random forcing: Ornstein-Uhlenbeck, compound Poisson, and a chaotic process

Now a random forcing on the four modes with wavenumbers $(1, 2)$ and $(2, 1)$, driven by three kinds of processes, with no noise:

In [ ]:
rmodes = fourier_modes(grid, [(1, 2), (2, 1)])
tspan = (0.0, 200.0)

ou = ou_path(4, tspan; θ = 0.5, s = 0.3, dt = 0.01, rng = Xoshiro(1))
cp = compound_poisson_path(4, tspan; rate = 0.2, θ = 0.2, jump = rng -> 0.3randn(rng), rng = Xoshiro(2))
lz = lorenz_path(tspan; timescale = 5.0, scale = 0.3)
lz4(t) = (x = lz(t); [x[1], x[2], x[3], 0.0])
nothing

In [ ]:
ts = 0:0.1:200
plot(
    plot(ts, reduce(hcat, [copy(ou(t)) for t in ts])', label = false, title = "Ornstein-Uhlenbeck", titlefont = 10),
    plot(ts, reduce(hcat, [copy(cp(t)) for t in ts])', label = false, title = "compound Poisson, with decay", titlefont = 10),
    plot(ts, reduce(hcat, [lz4(t) for t in ts])', label = false, title = "Lorenz-63", titlefont = 10, xlabel = "t"),
    layout = (3, 1), size = (700, 600)
)

A random forcing alone is a time-dependent forcing, so the deterministic solvers apply, path by path; for the compound Poisson process, with jumps, we give the jump times to the adaptive solver as `tstops`:

In [ ]:
runs = map([("Ornstein-Uhlenbeck", ou, (;)), ("compound Poisson", cp, (; tstops = jump_times(cp))), ("Lorenz-63", lz4, (;))]) do (name, η, kw)
    m = NSEModel(grid; ν = 1e-2, random_forcing = RandomForcing(rmodes, η))
    name => simulate(NSEProblem(m, zeros(grid.Nx, grid.Ny), tspan), Tsit5(); reltol = 1e-6, abstol = 1e-9,
        save_every = 0.5, spectrum = false, kw...)
end
plot([r.t for (_, r) in runs], [r.energy for (_, r) in runs], label = permutedims(first.(runs)),
    xlabel = "t", ylabel = "E", title = "Energy of flows driven by random forcings", titlefont = 10)

Both terms can be combined, with the stochastic schemes: here, the noise of the first example together with the Ornstein-Uhlenbeck forcing:

In [ ]:
model2 = NSEModel(grid; ν = 1e-2, noise, random_forcing = RandomForcing(rmodes, ou))
res2 = simulate(NSEProblem(model2, zeros(grid.Nx, grid.Ny), tspan; kind = :sde), StochasticETDRK4();
    dt = 0.01, save_every = 0.5, seed = 7)
plot(res2.t, res2.energy, xlabel = "t", label = "E", title = "Noise and Ornstein-Uhlenbeck forcing", titlefont = 10)

## Comparison of methods

To compare the methods, we use the flow of the first example (noise on the modes with $3 \leq |\boldsymbol{\kappa}| \leq 5$, $\varepsilon = 0.1$, $\nu = 10^{-2}$, $64 \times 64$) and the exact budgets $2\nu\langle Z\rangle = \varepsilon$ and $2\nu\langle P\rangle = \eta$ as references. Each method starts from the same state, at $t = 600$ after a warm-up from rest with the stochastic ETDRK4 scheme, and runs for 2000 time units, with independent noise. The tables give the relative imbalances $2\nu\langle Z\rangle/\varepsilon - 1$ and $2\nu\langle P\rangle/\eta - 1$ (corrected for the change of $E$ and $Z$ over the run), with 95% confidence intervals from 10 batch means, the time-averaged energy, the maximum CFL number for the fixed-step methods (with $k_{\max} = 21$) and the cost. These results were obtained once, on a 4-core Linux container, with each run single-threaded and four runs at a time, and are static: the code that produced them is in the folder [`benchmarks/stochastic/`](https://github.com/rmsrosa/2dnsepervortjl/tree/main/benchmarks/stochastic) of the repository. The timings are indicative only. In the pilot run, the flow was statistically steady after about 150 time units, with $\max(|u| + |v|) \approx 8$ and $\kappa_\sigma \approx 4.1$.

| Method | dt or tolerance | Status | Max CFL | Energy balance 2ν⟨Z⟩/ε - 1 | Enstrophy balance 2ν⟨P⟩/η - 1 | ⟨E⟩ | Nonlinear evaluations per time unit | Steps per time unit | Wall time per time unit (s) |
| :--- | ---: | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Stochastic ETDRK4 | dt = 0.025 | stable | 4.43 | +0.005 ± 0.011 | +0.003 ± 0.012 | 2.261 | 160 | 40 | 0.0104 |
| Stochastic ETDRK4 | dt = 0.02 | stable | 3.71 | +0.007 ± 0.012 | +0.006 ± 0.013 | 2.272 | 200 | 50 | 0.0121 |
| Stochastic ETDRK4 | dt = 0.01 | stable | 1.83 | +0.002 ± 0.013 | +0.002 ± 0.012 | 2.280 | 400 | 100 | 0.0234 |
| Stochastic ETDRK4 | dt = 0.005 | stable | 0.92 | -0.004 ± 0.012 | -0.005 ± 0.012 | 2.247 | 800 | 200 | 0.0485 |
| Split RK4 | dt = 0.025 | stable | 4.39 | -0.007 ± 0.013 | -0.006 ± 0.013 | 2.222 | 160 | 40 | 0.0116 |
| Split RK4 | dt = 0.02 | stable | 3.48 | -0.004 ± 0.019 | -0.003 ± 0.019 | 2.253 | 200 | 50 | 0.0130 |
| Split RK4 | dt = 0.01 | stable | 1.72 | +0.000 ± 0.019 | -0.002 ± 0.015 | 2.266 | 400 | 100 | 0.0252 |
| Split RK4 | dt = 0.005 | stable | 0.93 | +0.001 ± 0.014 | -0.003 ± 0.015 | 2.252 | 800 | 200 | 0.0522 |
| Split SSPRK3 | dt = 0.025 | blew up at t = 602 | – | – | – | – | – | – | – |
| Split SSPRK3 | dt = 0.02 | stable | 3.49 | +0.003 ± 0.011 | +0.002 ± 0.013 | 2.295 | 150 | 50 | 0.0105 |
| Split SSPRK3 | dt = 0.01 | stable | 1.78 | -0.010 ± 0.011 | -0.005 ± 0.009 | 2.229 | 300 | 100 | 0.0209 |
| Split SSPRK3 | dt = 0.005 | stable | 0.89 | -0.004 ± 0.011 | -0.001 ± 0.014 | 2.269 | 600 | 200 | 0.0415 |
| Stochastic exponential Euler | dt = 0.002 | blew up at t = 658 | – | – | – | – | – | – | – |
| Stochastic exponential Euler | dt = 0.001 | stable | 0.19 | +0.033 ± 0.009 | +0.052 ± 0.010 | 2.362 | 1000 | 1000 | 0.0888 |
| Stochastic exponential Euler | dt = 0.0005 | stable | 0.09 | +0.002 ± 0.011 | +0.015 ± 0.010 | 2.266 | 2000 | 2000 | 0.1816 |
| Euler-Maruyama | dt = 0.001 | stable | 0.18 | +0.045 ± 0.012 | +0.064 ± 0.010 | 2.377 | 1000 | 1000 | 0.0565 |
| Euler-Maruyama | dt = 0.0005 | stable | 0.09 | +0.023 ± 0.011 | +0.032 ± 0.011 | 2.352 | 2000 | 2000 | 0.1114 |
| SRA1 | tol = 1.0 | stable | – | -0.005 ± 0.015 | -0.005 ± 0.016 | 2.240 | – | 96 | 0.0132 |
| SRA1 | tol = 0.1 | stable | – | +0.000 ± 0.013 | -0.002 ± 0.011 | 2.281 | – | 1632 | 0.1977 |
| SOSRA | tol = 1.0 | stable | – | +0.008 ± 0.010 | +0.008 ± 0.008 | 2.287 | – | 158 | 0.0281 |

For the adaptive methods of StochasticDiffEq, `tol` is `reltol = abstol`. Their solver statistics do not count the evaluations of the drift, so only the steps are given.

![Energy balance](/assets/images/stochastic/energy_balance.png)

![Enstrophy balance](/assets/images/stochastic/enstrophy_balance.png)

### Discussion

* **All the stable runs sample the same statistics**, $\langle E\rangle \approx 2.22$ to $2.30$, except for the biased ones below, and the budgets close within the sampling error, about $\pm 1\%$ to $\pm 2\%$, for the stochastic ETDRK4, the split RK4 and SSPRK3 schemes, and SRA1 and SOSRA.
* **The schemes based on forward Euler for the advection, Euler-Maruyama and the stochastic exponential Euler, are biased**: Euler-Maruyama injects 2% to 6% too much energy and enstrophy (more for enstrophy) even with steps 25 to 50 times smaller than the others, and the stochastic exponential Euler scheme 3% to 5% at `dt = 0.001`, and still 1.5% too much enstrophy at `dt = 0.0005`, at 5 to 17 times the cost of the others. For Euler-Maruyama, the bias halves when the step is halved, as expected of a method of weak order one. Forward Euler amplifies the advected modes slightly at every step (its stability region does not contain the imaginary axis), which only the viscosity counteracts, and this shows up in the budgets. The stochastic exponential Euler scheme blew up at `dt = 0.002`, with a CFL number of only about 0.4.
* **The stochastic ETDRK4 and the split RK4 schemes** were stable up to the largest step tried, `dt = 0.025`, with a maximum CFL number of 4.4, and showed no detectable bias over a factor of 5 in the step. They are the cheapest, together with the split SSPRK3 scheme, which is cheaper per step but blew up at `dt = 0.025` (stable at `dt = 0.02`, CFL 3.5).
* **The adaptive methods SRA1 and SOSRA**, designed for additive noise, are impractical with usual tolerances: with `reltol = 1e-2` and `abstol = 1e-5`, SRA1 took about 280 000 steps per time unit (37 s per time unit, thousands of times more than the fixed-step schemes), its error control being dominated by the absolute errors in the many small coefficients. With `reltol = abstol = 1`, the steps are limited by stability rather than accuracy, about 100 per time unit, the cost is comparable to the fixed-step schemes, and the budgets close within the error; a tighter tolerance, 0.1, cost 15 times more with no measurable gain.

**Recommendation.** For advection-dominated flows driven by additive noise, the stochastic ETDRK4 scheme (or the split RK4 scheme) with a fixed step chosen from the CFL condition, as in [Time stepping for weakly turbulent flows](../../pages/Time stepping for weakly turbulent flows), with $\sigma \approx 2$ to $3$ relative to the largest velocities. The stochastic ETDRK4 scheme also integrates the viscous term exactly, so it handles stiff viscous terms as well. Euler-Maruyama and the stochastic exponential Euler scheme should be avoided for such flows. The comparison is for a single configuration, and differences in the statistics smaller than about 1% cannot be detected with these runs.